# Newman/Ratner full coverage review on Colab

This rerun aligns every eligible timed CHAT utterance in `5694MC` and renders the full recording with **CHAT transcript, independent WhisperX transcript, independent WhisperX word timing, and CHAT-guided word timing together**. Turns without a usable CHAT time link cannot be placed precisely; the report counts them.

Use a GPU runtime. Enter the Hugging Face token in the password widget below. Upload `5694MC.mp3` and `5694MC.cha` to `/content` if this is a new runtime. The previous pilot results remain in their separate output directory.


In [ ]:
from pathlib import Path
import os, subprocess, shutil, sys
repo = Path('/content/ytspeechpipeline')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/newman-quality-pilot',
                    'https://github.com/vladinsc/ytspeechpipeline.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin',
                    'codex/newman-quality-pilot'], check=True)
os.chdir(repo)
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
print('Repository:', repo)


In [ ]:
%pip install -q 'whisperx>=3.8.4,<3.9' 'praat-parselmouth>=0.4.3' 'numpy>=2.1,<3' ipywidgets


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then reconnect.'
print(torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
import ipywidgets as widgets
from IPython.display import display
password = widgets.Password(description='HF_TOKEN')
set_button = widgets.Button(description='Set token')
status = widgets.Output()
def set_token(_):
    if not password.value.strip():
        return
    os.environ['HF_TOKEN'] = password.value.strip()
    password.value = ''
    password.disabled = True
    set_button.disabled = True
    with status:
        print('Token set in this runtime (value hidden).')
set_button.on_click(set_token)
display(password, set_button, status)


In [ ]:
assert os.environ.get('HF_TOKEN'), 'Enter the token in the widget and click Set token first.'
# Confirm both files were uploaded to /content before running this cell.
relative_paths = {
    '5694MC.mp3': 'media/Interviews/18/5694MC.mp3',
    '5694MC.cha': 'transcripts/extracted/Interviews/18/5694MC.cha',
}
for name, relative in relative_paths.items():
    source = Path('/content') / name
    if not source.is_file():
        raise FileNotFoundError(f'Upload {name} to /content and rerun this cell.')
    destination = repo / 'data/staging/newman_ratner' / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(name, 'uploaded:', round(destination.stat().st_size / 1e6, 2), 'MB')


This run uses `medium.en`, int8, batch size 1, and **all eligible timed CHAT turns**. It writes to a new directory, so it will not resume into the 12-turn sample. It also transcribes and diarizes the full audio for the comparison video.


In [ ]:
source_id = 'NewmanRatner:18:5694MC:interview'
out_dir = repo / 'results/newman_quality_colab_full'
base_command = [sys.executable, 'tools/run_newman_quality_pilot.py',
                '--source-id', source_id, '--out-dir', str(out_dir),
                '--whisper-model', 'medium.en', '--compute-type', 'int8',
                '--batch-size', '1', '--turns-per-speaker', '0']
subprocess.run(base_command + ['--dry-run'], check=True, env=os.environ.copy())
subprocess.run(base_command, check=True, env=os.environ.copy())


In [ ]:
import json
work = out_dir / source_id.replace(':', '_')
report = json.loads((work / 'report.json').read_text(encoding='utf-8'))
print('Speaker coverage:', report['speaker_diarization']['speaker_coverage'])
print('Mapped speaker accuracy on covered speech:', report['speaker_diarization']['mapped_accuracy_on_covered_speech'])
print('Diagnostic ASR WER:', report['independent_asr']['diagnostic_wer'])
print('Aligned word coverage:', report['guided_alignment']['aligned_word_coverage'])
print('CHAT video inventory:', report['review_video_chat_inventory'])
print('Guided turns:', report['guided_alignment']['reviewed_turn_count'])
print('Review videos:', report['artifacts'].get('review_videos', []))
archive = shutil.make_archive('/content/newman_quality_colab_full', 'zip', root_dir=out_dir)
print('Download the results ZIP:', archive)
